In [1]:
import os
HOME = os.getcwd()
print(HOME)

C:\Users\Lenovo\OneDrive\Documents\Desktop\Uptoskills\Car-Parking-Slot-Ocuupancy-detection-model


In [2]:
import numpy as np
import cv2

In [3]:
camera = cv2.VideoCapture("parking1.mp4")


In [4]:
from ultralytics import YOLO

from IPython.display import display, Image

In [5]:
model=YOLO(r'30epoch_model\best.pt')


FileNotFoundError: [Errno 2] No such file or directory: '30epoch_model\\best.pt'

In [ ]:
file = open("slot_coordinates.txt")

print("[INFO] Loading parking coordinates ...")

lines = file.readlines()

lines = [line.strip() for line in lines]

total_parking_lots = len(lines)

parking_lot_coords = list()

for i in range(len(lines)):
    
    coords = lines[i].split()
    top_left = (int(coords[0]), int(coords[1]))
    top_right = (int(coords[2]), int(coords[3]))
    bottom_right = (int(coords[4]), int(coords[5]))
    bottom_left = (int(coords[6]), int(coords[7]))
    coord = np.array([top_left,top_right, bottom_right,bottom_left])
    parking_lot_coords.append(coord)
print(parking_lot_coords)

In [ ]:
while True:
    available_slot=len(parking_lot_coords)
    
    ret, frame = camera.read()

    if not ret:
        
        print("[ERROR] Failed to initialize camera.")
        cv2.destroyAllWindows()
        break

    if ret:
        frame=cv2.resize(frame,(1300,650))
        for i in range(len(parking_lot_coords)):

            cv2.polylines(frame,[np.array(parking_lot_coords[i],np.int32)],True,(0,255,0),2)
           
            
        results = model.predict(frame)
        result=results[0]
        for box in result.boxes:
            class_id = box.cls[0].item()
            if class_id==0.0:
                cord = box.xyxy[0].tolist()
                cord = [round(x) for x in cord]
                cx=int(cord[0]+cord[2])//2
                cy=int(cord[1]+cord[3])//2
                point=(cx,cy)
                
                for arr in parking_lot_coords:
                    if cv2.pointPolygonTest(np.array(arr,np.int32),((cx,cy)),False)>=0:
                        available_slot-=1   
                        
                        cv2.polylines(frame,[np.array(arr,np.int32)],True,(0,0,255),2)
                        
        print(available_slot)
        cv2.putText(frame,"Available Slots:"+str(available_slot),(50,25),cv2.FONT_HERSHEY_COMPLEX,1,(0,0,255),2)   


        cv2.imshow("Camera", frame)

        key=cv2.waitKey(1)

        if key%256 == 27:
            
            # ESC
            
            print("[INFO] Camera terminated.")
            cv2.destroyAllWindows()
            break

In [ ]:
import cv2

# Open the input video file
input_file = 'output_video.mp4'
cap = cv2.VideoCapture(input_file)

# Get the original video's properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define the output video file
output_file = 'final_output_video.mp4'
output_fps = 10.0

# Create a VideoWriter object to save the output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec
out = cv2.VideoWriter(output_file, fourcc, output_fps, (width, height))

# Variables for frame duplication
frame_count = 0
frame_duplicate = int(fps / output_fps)

# Read and write frames until the end of the video
while True:
    ret, frame = cap.read()

    if not ret:
        break

    frame_count += 1

    # Duplicate frames
    for _ in range(frame_duplicate):
        out.write(frame)

# Release the VideoCapture and VideoWriter objects
cap.release()
out.release()
